# AM5061 · Week 14 · Plant-level audit

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "glide", "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
    "lmtd", "effectiveness", "ntu_required", "exergy", "T0_REF", "P0_REF",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


# ------------------------------------------------- exchanger relations
def lmtd(dT1, dT2):
    """Log-mean temperature difference.

    Falls back to the arithmetic mean when the two ends are within 1% of each
    other, where the log form is numerically unstable and the two agree to
    better than 0.01% anyway.
    """
    dT1, dT2 = float(dT1), float(dT2)
    if dT1 <= 0 or dT2 <= 0:
        raise ValueError(
            f"temperature difference must be positive at both ends "
            f"(got {dT1:g} and {dT2:g}). A non-positive end means the streams "
            "cross, which no exchanger of this configuration can do."
        )
    if abs(dT1 - dT2) < 0.01*max(dT1, dT2):
        return 0.5*(dT1 + dT2)
    return (dT1 - dT2)/math.log(dT1/dT2)


def effectiveness(config, NTU, Cr):
    """Effectiveness for the standard configurations.

    config: 'counter', 'parallel', 'shell1'  (one shell pass, 2/4/... tube passes),
            'cross-both-unmixed' (approximate), 'cross-Cmax-mixed', 'cross-Cmin-mixed'

    Cr = C_min/C_max. Cr = 0 is the phase-change limit and every configuration
    collapses to the same expression, which is why boilers and condensers are
    easy and everything else is not.
    """
    if NTU < 0:
        raise ValueError("NTU cannot be negative")
    if not 0 <= Cr <= 1:
        raise ValueError(f"Cr must be between 0 and 1, got {Cr:g}")
    if Cr == 0:                       # phase change on one side
        return 1 - math.exp(-NTU)
    if config == "counter":
        if abs(Cr - 1) < 1e-12:
            return NTU/(1 + NTU)
        e = math.exp(-NTU*(1 - Cr))
        return (1 - e)/(1 - Cr*e)
    if config == "parallel":
        return (1 - math.exp(-NTU*(1 + Cr)))/(1 + Cr)
    if config == "shell1":
        r = math.sqrt(1 + Cr*Cr)
        e = math.exp(-NTU*r)
        return 2/(1 + Cr + r*(1 + e)/(1 - e))
    if config == "cross-both-unmixed":
        return 1 - math.exp((math.exp(-Cr*NTU**0.78) - 1)*NTU**0.22/Cr)
    if config == "cross-Cmax-mixed":
        return (1/Cr)*(1 - math.exp(-Cr*(1 - math.exp(-NTU))))
    if config == "cross-Cmin-mixed":
        return 1 - math.exp(-(1 - math.exp(-Cr*NTU))/Cr)
    raise ValueError(f"unknown configuration {config!r}")


def ntu_required(config, eps, Cr, hi=200.0):
    """Invert effectiveness() for NTU. Design direction, rather than rating."""
    eps_max = effectiveness(config, hi, Cr)
    if eps >= eps_max:
        raise ValueError(
            f"effectiveness {eps:g} is unreachable for {config} at Cr={Cr:g}; "
            f"the limit as NTU->infinity is {eps_max:.6f}. "
            "Change the configuration or accept less."
        )
    return solve(lambda n: effectiveness(config, n, Cr) - eps, 1.0,
                 bracket=(1e-9, hi))


# --------------------------------------------------------------- exergy
T0_REF, P0_REF = 303.15, 101325.0      # 30 C, sea level: the Chennai dead state


def exergy(st, T0=T0_REF, p0=P0_REF):
    """Specific flow exergy, J/kg:  (h - h0) - T0*(s - s0).

    The dead state is the ambient the plant actually sits in, so it is a
    DESIGN CHOICE, not a constant. Report which one you used - a Chennai
    dead state and a European one give different answers for the same plant.
    """
    ref = State(st.fluid, T=T0, P=p0)
    return (st.h - ref.h) - T0*(st.s - ref.s)


---
## The case

Take an earlier case and answer the only two questions a plant manager asks:
**what does it cost, and where is the loss?**

Deliverable **D-14**: Sankey diagram, exergy destruction table, levelised cost
and carbon intensity for one case of your choice. The worked example here uses
**Week 10's bagasse cogeneration plant**; swap in your own.

### Why energy accounting is not enough

A first-law balance says energy is conserved, so it can never tell you where the
*opportunity* went. Exergy can. A boiler that is 88% efficient on energy is
typically **under 30% efficient on exergy**, and that gap is where the design
work actually is.


## 1. The plant, from Week 10

In [ ]:
import am5061 as am
import numpy as np, matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI
am.style_plots()

T0, p0 = am.K(30), 101325.0          # Chennai dead state. A DESIGN CHOICE.
print(f"  dead state: {am.C(T0):.0f} C, {p0/1e3:.3f} kPa")
print("  Every exergy number below is relative to this. Report it, always.\n")

# plant streams, from the Week 10 balance at 100% crush
m_steam = 20.255                     # kg/s through the turbine
p_hp, T_hp = 87e5, am.K(515)
p_bp       = 3.5e5
LHV_bag    = 7.5e6                   # J/kg, bagasse at 50% moisture
m_bag      = 43.75*1000/3600         # kg/s

st_hp  = am.State("Water", P=p_hp, T=T_hp)
st_bp  = am.State("Water", P=p_bp, H=2801.2e3)
st_fw  = am.State("Water", P=p_hp, T=am.K(105))
Q_fuel = m_bag*LHV_bag
W_turb = m_steam*(st_hp.h - st_bp.h)*0.96
Q_proc = m_steam*(st_bp.h - PropsSI("H","P",p_bp,"Q",0,"Water"))

print(f"  fuel input     {Q_fuel/1e6:8.2f} MW")
print(f"  shaft/electric {W_turb/1e6:8.2f} MW")
print(f"  process heat   {Q_proc/1e6:8.2f} MW")
print(f"  first-law utilisation {(W_turb+Q_proc)/Q_fuel*100:.1f}%")


## 2. Exergy — the second-law picture

Flow exergy is `(h − h₀) − T₀(s − s₀)`. For the fuel we use a chemical exergy
approximated as **1.15 × LHV** for a solid biomass, which is the usual
engineering shortcut.


In [ ]:
ex_hp = am.exergy(st_hp, T0, p0)
ex_bp = am.exergy(st_bp, T0, p0)
ex_fw = am.exergy(st_fw, T0, p0)
Ex_fuel = 1.15*Q_fuel                      # chemical exergy of bagasse

Ex_steam_hp = m_steam*ex_hp
Ex_steam_bp = m_steam*ex_bp
Ex_fw_in    = m_steam*ex_fw

# component by component
Ex_d_boiler  = Ex_fuel + Ex_fw_in - Ex_steam_hp
Ex_d_turbine = Ex_steam_hp - Ex_steam_bp - W_turb
Ex_d_process = Ex_steam_bp - m_steam*am.exergy(am.State("Water",P=p_bp,Q=0), T0, p0)

comp = [
    {"component": "Boiler (combustion + heat transfer)", "exergy in, MW": Ex_fuel/1e6,
     "exergy destroyed, MW": Ex_d_boiler/1e6},
    {"component": "Turbine + generator", "exergy in, MW": Ex_steam_hp/1e6,
     "exergy destroyed, MW": Ex_d_turbine/1e6},
    {"component": "Process heat delivery", "exergy in, MW": Ex_steam_bp/1e6,
     "exergy destroyed, MW": Ex_d_process/1e6},
]
Ex_out_useful = W_turb + (Ex_steam_bp - m_steam*am.exergy(am.State("Water",P=p_bp,Q=0),T0,p0))
for c in comp:
    c["% of fuel exergy"] = 100*c["exergy destroyed, MW"]*1e6/Ex_fuel

print(f"  fuel exergy            {Ex_fuel/1e6:8.2f} MW")
print(f"  exergy in HP steam     {Ex_steam_hp/1e6:8.2f} MW")
print(f"{'component':<40}{'destroyed MW':>14}{'% of fuel':>12}")
for c in comp:
    print(f"{c['component']:<40}{c['exergy destroyed, MW']:14.3f}{c['% of fuel exergy']:12.1f}")
eta_I  = (W_turb + Q_proc)/Q_fuel
eta_II = (W_turb + (Ex_steam_bp - m_steam*am.exergy(am.State("Water",P=p_bp,Q=0),T0,p0)))/Ex_fuel
print(f"\n  first-law  efficiency  {eta_I*100:6.1f}%")
print(f"  second-law efficiency  {eta_II*100:6.1f}%")
print("\n  The boiler dominates. It always does: burning a fuel at 1200 C to")
print("  raise steam at 515 C throws away most of the fuel's quality before")
print("  any equipment downstream gets a chance to be inefficient.")


## 3. The Sankey

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.4, 4.6))

# --- energy Sankey, as a stacked bar (readable, and honest about proportions)
labels_E = ["shaft power", "process heat", "stack + losses"]
vals_E   = [W_turb/1e6, Q_proc/1e6, (Q_fuel - W_turb - Q_proc)/1e6]
left = 0
for lab, v, col in zip(labels_E, vals_E, [am.ORANGE, am.BLUE, am.MUTED]):
    a1.barh(0, v, left=left, color=col, edgecolor="white", label=f"{lab}  {v:.1f} MW")
    if v/sum(vals_E) > 0.06:
        a1.text(left + v/2, 0, f"{100*v/sum(vals_E):.0f}%", ha="center",
                va="center", color="white", fontweight="bold")
    left += v
a1.set_yticks([]); a1.set_xlabel("MW"); a1.set_title(f"ENERGY: {Q_fuel/1e6:.1f} MW of fuel in")
a1.legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5,-0.18), ncol=1)

# --- exergy Sankey
labels_X = ["shaft power", "useful process exergy",
            "destroyed in boiler", "destroyed in turbine", "destroyed elsewhere"]
X_proc_useful = Ex_steam_bp - m_steam*am.exergy(am.State("Water",P=p_bp,Q=0),T0,p0)
vals_X = [W_turb/1e6, X_proc_useful/1e6, Ex_d_boiler/1e6, Ex_d_turbine/1e6,
          max((Ex_fuel - W_turb - X_proc_useful - Ex_d_boiler - Ex_d_turbine)/1e6, 0)]
left = 0
for lab, v, col in zip(labels_X, vals_X,
                       [am.ORANGE, am.BLUE, "#B03A2E", "#D98880", am.MUTED]):
    a2.barh(0, v, left=left, color=col, edgecolor="white", label=f"{lab}  {v:.1f} MW")
    if v/sum(vals_X) > 0.06:
        a2.text(left + v/2, 0, f"{100*v/sum(vals_X):.0f}%", ha="center",
                va="center", color="white", fontweight="bold")
    left += v
a2.set_yticks([]); a2.set_xlabel("MW"); a2.set_title(f"EXERGY: {Ex_fuel/1e6:.1f} MW in")
a2.legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5,-0.18), ncol=1)
plt.tight_layout(); plt.show()
print("  Same plant, two accounts. The energy picture says 'mostly useful'.")
print("  The exergy picture says 'mostly destroyed, and here is where'.")
print("  Only the second one tells you what to redesign.")


## 4. The 4R sequence, driven by the exergy map

**Reduce · Reuse · Recycle · Replace** — applied in that order, and targeted by
the exergy table rather than by intuition. The largest destruction term is where
you look first, but only if something can actually be done about it.


In [ ]:
actions = [
 {"target": "Boiler combustion irreversibility", "R": "Replace",
  "action": "Raise steam pressure/temperature, or gasify rather than burn",
  "exergy at stake, MW": Ex_d_boiler/1e6,
  "realistic recovery, %": 10,
  "note": "Largest term, hardest to touch. Combustion irreversibility is set by flame temperature."},
 {"target": "Vented exhaust steam", "R": "Reduce",
  "action": "Match steam raising to process demand; trim boiler output",
  "exergy at stake, MW": 14.58*1000/3600*ex_bp/1e6,
  "realistic recovery, %": 80,
  "note": "Cheapest win on the list. Pure waste, no thermodynamic obstacle."},
 {"target": "Turbine irreversibility", "R": "Replace",
  "action": "Higher-efficiency turbine on next overhaul",
  "exergy at stake, MW": Ex_d_turbine/1e6,
  "realistic recovery, %": 15,
  "note": "Capital-intensive, incremental."},
]
print(f"{'target':<36}{'R':<9}{'at stake MW':>13}{'recoverable MW':>16}")
for a in actions:
    rec = a["exergy at stake, MW"]*a["realistic recovery, %"]/100
    a["recoverable, MW"] = rec
    print(f"{a['target']:<36}{a['R']:<9}{a['exergy at stake, MW']:13.2f}{rec:16.3f}")
print("\n  Note the ordering. The BIGGEST destruction term is not the best")
print("  opportunity - the vented steam is. Exergy tells you where the loss is;")
print("  engineering judgement tells you which losses you can actually get back.")


## 5. Techno-economics

In [ ]:
CAPEX_per_kW = 65000.0      # Rs/kW installed, Indian bagasse cogen, order of magnitude
LIFE, DISC   = 20, 0.10     # years, discount rate
OPEX_frac    = 0.04         # of capex per year
CUF          = 0.45         # capacity utilisation: a sugar mill runs seasonally
TARIFF       = 4.50         # Rs/kWh, export tariff
GRID_CO2     = 0.71         # kg CO2/kWh, Indian grid average

P_kW   = W_turb/1e3
capex  = P_kW*CAPEX_per_kW
crf    = DISC*(1+DISC)**LIFE/((1+DISC)**LIFE - 1)
annual_kWh = P_kW*8760*CUF
lcoe   = (capex*crf + capex*OPEX_frac)/annual_kWh

print(f"  installed capacity      {P_kW:10.0f} kW")
print(f"  capex                   {capex/1e7:10.2f} crore Rs")
print(f"  capital recovery factor {crf:10.4f}")
print(f"  annual generation       {annual_kWh/1e6:10.2f} GWh   (CUF {CUF:.0%})")
print(f"\n  levelised cost          {lcoe:10.2f} Rs/kWh")
print(f"  export tariff           {TARIFF:10.2f} Rs/kWh")
print(f"  margin                  {TARIFF-lcoe:10.2f} Rs/kWh")
print(f"  simple payback          {capex/((TARIFF-lcoe)*annual_kWh):10.2f} years"
      if TARIFF > lcoe else "  NOT VIABLE at this tariff")


In [ ]:
fig, ax = plt.subplots()
cufs = np.linspace(0.20, 0.90, 60)
for cap in (50000, 65000, 80000):
    l = [(P_kW*cap*crf + P_kW*cap*OPEX_frac)/(P_kW*8760*c) for c in cufs]
    ax.plot(cufs*100, l, lw=2.2, label=f"{cap/1000:.0f}k Rs/kW installed")
ax.axhline(TARIFF, color="#B03A2E", ls="--", lw=1.8)
ax.text(22, TARIFF+0.15, f"export tariff {TARIFF:.2f} Rs/kWh", color="#B03A2E", fontsize=9)
ax.axvline(CUF*100, color=am.MUTED, ls=":")
ax.set_xlabel("capacity utilisation factor  (%)"); ax.set_ylabel("levelised cost  (Rs/kWh)")
ax.set_title("Seasonality, not efficiency, decides whether cogeneration pays")
ax.set_ylim(0, 12); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
print("  A sugar mill crushes for perhaps 160 days a year. That CUF, not the")
print("  turbine efficiency, is what usually kills or saves the business case.")
print("  This is why off-season operation on biomass is worth so much.")


## 6. Carbon

In [ ]:
co2_avoided = annual_kWh*GRID_CO2/1000        # tonnes/yr
print(f"  displaced grid electricity  {annual_kWh/1e6:8.2f} GWh/yr")
print(f"  grid emission factor        {GRID_CO2:8.2f} kg CO2/kWh")
print(f"  CO2 avoided                 {co2_avoided:8.0f} t/yr")
print(f"  carbon intensity of output  {0.0:8.2f} kg CO2/kWh  (bagasse counted as biogenic)")
print("\n  MARGINAL vs AVERAGE. The 0.71 figure is the grid AVERAGE. What you")
print("  actually displace at 2 a.m. is the MARGINAL plant, usually coal, with")
print("  a higher factor. Using the average understates the benefit; using coal")
print("  overstates it. State which you used and why - this is where carbon")
print("  accounting arguments are won and lost.")


## 7. The audit workbook

In [ ]:
econ = [{"CUF, %": float(c*100),
         "LCOE, Rs/kWh": (capex*crf + capex*OPEX_frac)/(P_kW*8760*c),
         "annual GWh": P_kW*8760*c/1e6,
         "margin, Rs/kWh": TARIFF - (capex*crf + capex*OPEX_frac)/(P_kW*8760*c)}
        for c in np.arange(0.20, 0.91, 0.05)]

path = am.to_excel("AM5061_D14_PlantAudit.xlsx",
    {"Exergy destruction": comp, "4R actions": actions, "Economics vs CUF": econ},
    title="AM5061 D-14 . Plant audit of the bagasse cogeneration case",
    summary=[("Dead state", f"{am.C(T0):.0f} C / {p0/1e3:.1f} kPa", ""),
             ("Fuel energy input", Q_fuel/1e6, "MW"),
             ("Fuel exergy input", Ex_fuel/1e6, "MW"),
             ("Shaft power", W_turb/1e6, "MW"),
             ("Process heat", Q_proc/1e6, "MW"),
             ("First-law efficiency", eta_I*100, "%"),
             ("Second-law efficiency", eta_II*100, "%"),
             ("Largest exergy destruction", "boiler", ""),
             ("LCOE", lcoe, "Rs/kWh"), ("Export tariff", TARIFF, "Rs/kWh"),
             ("CO2 avoided", co2_avoided, "t/yr")],
    sources=[("Steam properties", "CoolProp, IAPWS-95"),
             ("Dead state", "Chennai ambient 30 C, 101.325 kPa - a DESIGN CHOICE"),
             ("Bagasse LHV", "7.5 MJ/kg at 50% moisture - TYPICAL, confirm for the mill"),
             ("Bagasse chemical exergy", "1.15 x LHV - standard approximation for solid biomass"),
             ("Capex", "65,000 Rs/kW installed - ORDER OF MAGNITUDE, get vendor quotes"),
             ("Grid emission factor", "0.71 kg CO2/kWh, Indian grid average - AVERAGE not marginal"),
             ("Plant data", "AM5061 D-10, Week 10")])
print("written:", path)


## What to hand in

1. Both Sankeys — energy and exergy — for **your** chosen case, side by side.
2. The exergy destruction table, ranked, with the dead state stated.
3. Your 4R action list, ordered by **recoverable** exergy rather than by
   destroyed exergy, with the difference explained.
4. LCOE, the tariff comparison, and the CUF sensitivity.
5. Carbon intensity, stating whether you used the average or marginal factor.
6. The workbook.

**Every cost and emission figure in this notebook is an order-of-magnitude
placeholder.** They are labelled as such on the Sources sheet. A techno-economic
claim built on unsourced numbers is worth nothing; get quotes, cite the CEA
factor you use, and state the year.
